# Transform JSON data into CSV
1. Download raw JSON data from GCS bucket
2. Transform the data
3. Export the data as CSV

In [7]:
from google.cloud import storage

# 1. Set up your Google Cloud Storage client
storage_client = storage.Client()

# 2. Specify your bucket name and file path prefix
bucket_name = 'leverageai-sandbox-data'  # Replace with your actual bucket name
file_path_prefix = 'raw/source_name:fantasyfootballhub_predictions/source_date:2024-08-02/'  # Replace with the path to your file in the bucket

# List files (with optional prefix)
bucket = storage_client.bucket(bucket_name)
# blobs = bucket.list_blobs(prefix=file_path_prefix)
blobs = bucket.list_blobs(match_glob='raw/source_name:fantasyfootballhub_predictions/source_date:2024-08-02/*.json')

# 3. Download the files
for n,blob in enumerate(blobs):
    local_file_path = f'./.data/data{f"{n:03}"}.json'
    blob.download_to_filename(local_file_path)
    print(f"Downloaded {blob.name} to {local_file_path}")


Downloaded raw/source_name:fantasyfootballhub_predictions/source_date:2024-08-02/fantasyfootballhub_predictions_def_20240802.json to ./.data/data000.json
Downloaded raw/source_name:fantasyfootballhub_predictions/source_date:2024-08-02/fantasyfootballhub_predictions_fwd_20240802.json to ./.data/data001.json
Downloaded raw/source_name:fantasyfootballhub_predictions/source_date:2024-08-02/fantasyfootballhub_predictions_gk_20240802.json to ./.data/data002.json
Downloaded raw/source_name:fantasyfootballhub_predictions/source_date:2024-08-02/fantasyfootballhub_predictions_mid_20240802.json to ./.data/data003.json


In [35]:
import json
import pandas as pd

json_filepaths = [
    '/home/jon/workbench/github/leverageai/fpl/cloud_functions/process_fantasyfootballhub_predictions/scratch/.data/data000.json',
    '/home/jon/workbench/github/leverageai/fpl/cloud_functions/process_fantasyfootballhub_predictions/scratch/.data/data001.json',
    '/home/jon/workbench/github/leverageai/fpl/cloud_functions/process_fantasyfootballhub_predictions/scratch/.data/data002.json',
    '/home/jon/workbench/github/leverageai/fpl/cloud_functions/process_fantasyfootballhub_predictions/scratch/.data/data003.json'
]

input_dfs = []

for json_filepath in json_filepaths:
    # Load json file
    with open(json_filepath) as f:
        data = json.load(f)

    # Extract raw data into a Pandas DataFrame
    input_dfs.append(pd.DataFrame(data))

# Concatenate dataframes
raw_df = pd.concat(input_dfs, axis=0, ignore_index=True)

# Print the DataFrame
raw_df.head()


,web_name,code,team,now_cost,status,position_id,chance_next_round,player,results,predictions,...,position,club,fplreview,range_prediction,range_goals,range_assists,range_cs,range_returns,range_value,fpl_id
0,Alexander-Arnold,169187,"{'code_name': 'LIV', 'code': '14'}",70,a,2,None,"{'id': 952, 'code': 169187, 'first_name': 'Tre...","[{'gw': 23, 'xa': 0.03, 'opp': ['ars'], 'mins'...","[{'gw': 1, 'opp': [['ips', 'Ipswich (A)', 2]],...",...,Defender,Liverpool,"{'id': 138179, 'massive_points': None, 'io_poi...",20.287926,0.343465,1.427894,1.461045,3.232405,0.438854,311
1,Gvardiol,477424,"{'code_name': 'MCI', 'code': '43'}",60,a,2,None,"{'id': 2258, 'code': 477424, 'first_name': 'Jo...","[{'gw': 23, 'xa': 0.07, 'opp': ['bre'], 'mins'...","[{'gw': 1, 'opp': [['che', 'Chelsea (A)', 3]],...",...,Defender,Man City,"{'id': 140864, 'massive_points': None, 'io_poi...",19.769406,0.668342,0.394330,1.739936,2.802608,0.490392,350
2,Gabriel,226597,"{'code_name': 'ARS', 'code': '3'}",60,a,2,None,"{'id': 1532, 'code': 226597, 'first_name': 'Ga...","[{'gw': 23, 'xa': 0.02, 'opp': ['LIV'], 'mins'...","[{'gw': 1, 'opp': [['WOL', 'Wolves (H)', 3]], ...",...,Defender,Arsenal,"{'id': 136925, 'massive_points': None, 'io_poi...",19.211573,0.472925,0.188752,1.917765,2.579441,0.467149,3
3,White,198869,"{'code_name': 'ARS', 'code': '3'}",65,a,2,None,"{'id': 697, 'code': 198869, 'first_name': 'Ben...","[{'gw': 23, 'xa': 0.0, 'opp': ['LIV'], 'mins':...","[{'gw': 1, 'opp': [['WOL', 'Wolves (H)', 3]], ...",...,Defender,Arsenal,"{'id': 140048, 'massive_points': None, 'io_poi...",19.114263,0.279118,0.508821,1.917765,2.705704,0.427472,24
4,Calafiori,466075,"{'code_name': 'ARS', 'code': '3'}",60,a,2,None,"{'id': 2485, 'code': 466075, 'first_name': 'Ri...","[{'gw': 23, 'xa': 0, 'opp': ['LIV'], 'mins': 0...","[{'gw': 1, 'opp': [['WOL', 'Wolves (H)', 3]], ...",...,Defender,Arsenal,"{'id': 158813, 'massive_points': None, 'io_poi...",16.934314,0.133293,0.273778,1.917765,2.324836,0.372263,578


In [26]:
raw_df.columns

Index(['web_name', 'code', 'team', 'now_cost', 'status', 'position_id',
       'chance_next_round', 'player', 'results', 'predictions',
       'this_gameweek', 'season_prediction', 'season_prediction_avg',
       'search_term', 'position', 'club', 'fplreview', 'range_prediction',
       'range_goals', 'range_assists', 'range_cs', 'range_returns',
       'range_value', 'fpl_id'],
      dtype='object')

In [20]:
raw_df['fplreview'][0]

{'id': 138179,
 'massive_points': None,
 'io_points': None,
 'review_xg': None,
 'actual_points': None,
 'predict_pts90': 5.028483399275597,
 'predict_pts60': 0.0,
 'predict_xg90': None,
 'predict_g90': 0.09035433263022431,
 'predict_a90': 0.34182299861175386,
 'predict_bon90': 0.0,
 'predict_cs90': 0.3590962136743956,
 'predict_xmins': 86.0,
 'predict_points': 4.952982288330399,
 'predict_points_beta': None,
 'predict_apoints': 2.954928276305771,
 'predict_status': 'a',
 'predict_fitness': 1.0,
 'rank': None,
 'capt_bonus': 1.0,
 'predict_points_capt': None,
 'predict_xg': 0.08633858451332546,
 'predict_xg_beta': None,
 'predict_xa': 0.32663086534012037,
 'predict_xa_beta': None,
 'predict_cs': 0.3590962136743956,
 'predict_returns': 0.7720656635278415,
 'played': False,
 'mins': None,
 'opp': None,
 'any_goal': 0.08639259300350155,
 'any_assist': 0.2895260515443705,
 'any_cs': 0.3590962136743956,
 'any_return': 0.5839930299384171,
 'xg_adjust': 1.0,
 'xa_adjust': 1.0,
 'x_finish': 0.

In [43]:
from datetime import datetime

# Format output dataframe
output_df = pd.concat(
    [
        raw_df.loc[:,['web_name', 'code', 'now_cost', 'status', 'position_id',
       'chance_next_round', 'season_prediction', 'season_prediction_avg',
       'search_term', 'position', 'club', 'range_prediction',
       'range_goals', 'range_assists', 'range_cs', 'range_returns',
       'range_value', 'fpl_id']],
        pd.json_normalize(raw_df['team']).rename(columns={"code_name":"team_code_name","code":"team_code"}),
        pd.json_normalize(raw_df['player']).loc[:,['id','hub_owned','elite_owned','elite_weight']].rename(columns={"id":"player_id","hub_owned":"player_hub_owned","elite_owned":"player_elite_owned","elite_weight":"player_elite_weight"}),
        pd.json_normalize(raw_df['this_gameweek']).loc[:,['xmins','any_cs','any_goal','any_assist','any_return']].rename(columns={"xmins":"this_gameweek_xmins","any_cs":"this_gameweek_any_cs","any_goal":"this_gameweek_any_goal","any_assist":"this_gameweek_any_assist","any_return":"this_gameweek_any_return"})
    ],
    axis=1
)

# Save to CSV
output_df.to_csv(
    f'.data/{datetime.now().strftime("%Y%m%d%H%M%S")}.csv',
    index=False)